# AirfRANS GNN Surrogate — Kaggle GPU setup

Fallback for when Colab's free GPU quota runs out -- separate quota pool
(~30 hrs/week of P100 or T4x2). No Drive-equivalent live mount here: Kaggle
persists across sessions via **Datasets** (read-only, attached to a session)
and a notebook's own **Output** (from "Save Version"), not a synced folder.

Before running: Settings (right sidebar) > Accelerator > GPU, and
Internet > On (needed for git clone / pip install / dataset download).

## 0. This run starts fresh, deliberately -- not resuming

Earlier runs (Colab epoch 24 -> Kaggle epoch 29 -> ... -> epoch 99) used a
plain, uniformly-weighted loss. Evaluation on the official test set found
that training *improved* raw field accuracy all the way to epoch 99 but
*dramatically worsened* drag prediction past epoch 54 -- a real, measured
regression (see `notebooks/compare_checkpoints.py` and project history).

This run switches to a distance-weighted loss (`src/train.py`'s
`TrainModule`, nodes near the wall weighted up to 20x, since drag depends on
near-wall velocity gradients) to address that specifically. The old
checkpoints are architecture-compatible (same model shape) but were trained
under the *old* loss -- resuming from one would silently continue training a
model already shaped by the objective this run is trying to fix, corrupting
any clean before/after comparison. So: new checkpoint path below, auto-resume
finds nothing there, training starts at epoch 0 on purpose.

In [ ]:
# New, distinct checkpoint path -- keeps this run's checkpoints separate from
# the old (differently-trained) ones, so auto-resume in the training cell
# below finds nothing here and starts fresh at epoch 0, deliberately.
CHECKPOINT_PATH = "/kaggle/working/meshgraphnet_weighted.ckpt"

import glob
import os

existing = glob.glob(os.path.join(os.path.dirname(CHECKPOINT_PATH), "mgn-epoch=*.ckpt"))
print("existing checkpoints at this path (should be empty for a fresh start):", existing)

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 1. Get the repo and install dependencies

Same as Colab -- Kaggle also ships CUDA-enabled torch preinstalled, and
`torch_geometric` installs as pure Python (no `torch-scatter`/`torch-sparse`
needed, confirmed working on both machines already).

In [ ]:
REPO_URL = "https://github.com/Revanthkr1/airfrans-gnn-surrogate.git"

!git clone $REPO_URL repo
%cd repo
!pip install -q torch_geometric lightning airfrans pyvista

## 2. Download + preprocess, one case at a time

`af.dataset.download(unzip=True)` needs the zip (~9.34GB) *and* the fully
extracted dataset (~15GB) on disk simultaneously to extract everything at
once -- that alone exceeded this session's disk quota (`OSError: No space
left on device`, mid-extraction, before preprocessing even started).

Instead: download just the zip, then extract + cache one case at a time,
deleting each case's raw files immediately after caching. Peak disk usage
stays at roughly (zip + one case + the cache built so far) instead of
(zip + the entire raw dataset) at once. `manifest.json` doesn't need
extracting from the zip either -- it's committed to the repo at
`data/manifest.json`.

In [ ]:
from src.data import split_names
from src.preprocess import download_zip_only, stream_preprocess_from_zip

DATA_ROOT = "data"
WORK_DIR = "/kaggle/working/raw"  # transient extraction scratch, one case at a time
CACHE_DIR = "/kaggle/working/cache/full"
MANIFEST_DIR = "data"  # data/manifest.json is committed to the repo -- always present

train_names = split_names(MANIFEST_DIR, task="full", train=True)
zip_path = download_zip_only(DATA_ROOT)
stream_preprocess_from_zip(zip_path, train_names, CACHE_DIR, WORK_DIR)

import os
os.remove(zip_path)  # done with it -- frees ~9.34GB back
print(f"cached {len(os.listdir(CACHE_DIR))}/{len(train_names)} training cases")

## 3. Train, fresh, with the new distance-weighted loss

No `resume_from_checkpoint` passed -- `CHECKPOINT_PATH` (section 0) is a new,
empty directory, so auto-detect finds nothing and this starts at epoch 0 on
purpose (see section 0 for why resuming the old checkpoint would be wrong
here). Everything else is unchanged from the last run: `batch_size=1` +
`accumulate_grad_batches=4` (OOM headroom), `precision="16-mixed"`,
`checkpoint_every_n_epochs=5`.

Once this finishes (or periodically during it), use
`notebooks/compare_checkpoints.py` locally to check whether the new loss
actually fixed the epoch-54-vs-99 drag regression, the same way that
regression was originally found -- don't just trust the final epoch.

**To persist past this session**: click "Save Version" when done (or
periodically) -- that's Kaggle's equivalent of Drive surviving a disconnect.

In [ ]:
from src.train import main as train_main

train_main(
    dataset_root=MANIFEST_DIR,
    cache_dir=CACHE_DIR,
    stats_path="data/norm_stats.npz",
    checkpoint_path=CHECKPOINT_PATH,
    max_epochs=100,
    batch_size=1,
    accumulate_grad_batches=4,
    n_val=80,
    checkpoint_every_n_epochs=5,
    num_workers=2,
    precision="16-mixed",
)